<a href="https://colab.research.google.com/github/mf2056/Dissertation/blob/main/Baseline_ResUNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras import layers, models

In [ ]:
# Load the data
X_train = np.load('/content/drive/MyDrive/X_train.npy')
Y_train_mask = np.load('/content/drive/MyDrive/Y_train_mask.npy')
X_test = np.load('/content/drive/MyDrive/X_test.npy')
Y_test_mask = np.load('/content/drive/MyDrive/Y_test_mask.npy')

In [ ]:
import numpy as np
import tensorflow as tf

# Define the ESA class map
class_map = {
    10: 0, #(Tree cover, "#006400")
    20: 1, #(Shrubland, "#ffbb22")
    30: 2, #(Grassland, "#ffff4c")
    40: 3, #(Cropland, "#f096ff")
    60: 3, #(Bare / Sparse vegetation, "#b4b4b4")  Combined with Cropland
    50: 4, #(Built-up, "#fa0000")
    80: 5, #(Permanent water bodies, "#0064ff")
    90: 5, #(Herbaceous wetland, "#0096a0")   Combined with Water Bodies
}

# map the high ESA numbers to 0, 1, 2, 3, 4, 5 across all pixels
lut = np.full(91, -1, dtype=np.int32)
for old_id, new_id in class_map.items():
    lut[old_id] = new_id

# Apply mapping to the whole 3D array (Spatial Mapping)
Y_train_ready = lut[Y_train_mask]
Y_test_ready = lut[Y_test_mask]

print(f"Unique IDs in merged train mask: {np.unique(Y_train_ready)}")

Unique IDs in merged train mask: [0 1 2 3 4 5]


In [ ]:
num_classes = 6
Y_train_cat = tf.keras.utils.to_categorical(Y_train_ready, num_classes=num_classes)
Y_test_cat = tf.keras.utils.to_categorical(Y_test_ready, num_classes=num_classes)

print(f"X_train shape: {X_train.shape}") # no.of bands(7)
print(f"Y_train shape: {Y_train_cat.shape}") # no.of classes(6)

X_train shape: (639, 256, 256, 7)
Y_train shape: (639, 256, 256, 6)


In [ ]:
# Converting for RAM efficiency

X_train = X_train.astype('float32')
Y_train_cat = Y_train_cat.astype('float32')
X_test = X_test.astype('float32')
Y_test_cat = Y_test_cat.astype('float32')

In [ ]:
import tensorflow.keras.backend as K

def residual_block(x, filters, dropout_rate=0.3):
    shortcut = x

    # First Convolution
    x = layers.Conv2D(filters, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Second Convolution
    x = layers.Conv2D(filters, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)

    # Add Dropout BEFORE the addition to prevent memorization
    x = layers.Dropout(dropout_rate)(x)

    # Adjust shortcut dimensions if necessary
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, (1, 1), padding='same')(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    # The Addition
    x = layers.add([x, shortcut])
    x = layers.Activation('relu')(x)
    return x

def build_res_unet(input_shape=(256, 256, 7), num_classes=6):
    inputs = layers.Input(input_shape)

    # Encoder
    s1 = residual_block(inputs, 32)
    p1 = layers.MaxPooling2D((2, 2))(s1)

    s2 = residual_block(p1, 64)
    p2 = layers.MaxPooling2D((2, 2))(s2)

    s3 = residual_block(p2, 128)
    p3 = layers.MaxPooling2D((2, 2))(s3)

    # Bridge
    b1 = residual_block(p3, 256)

    # Decoder
    u1 = layers.Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(b1)
    u1 = layers.concatenate([u1, s3])
    d1 = residual_block(u1, 128)

    u2 = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(d1)
    u2 = layers.concatenate([u2, s2])
    d2 = residual_block(u2, 64)

    u3 = layers.Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same')(d2)
    u3 = layers.concatenate([u3, s1])
    d3 = residual_block(u3, 32)

    outputs = layers.Conv2D(num_classes, (1, 1), activation='softmax')(d3)
    return models.Model(inputs, outputs)

model = build_res_unet()
print("ResU-Net Built successfully!")

ResU-Net Built successfully!


In [ ]:
import tensorflow.keras.backend as K
import tensorflow as tf

def weighted_categorical_crossentropy(weights, label_smoothing=0.1):
    weights = K.variable(weights)

    def loss(y_true, y_pred):
        # Apply Label Smoothing manually
        num_classes = K.cast(K.shape(y_true)[-1], y_true.dtype)
        smooth_y_true = y_true * (1.0 - label_smoothing) + (label_smoothing / num_classes)

        # Standard Categorical Crossentropy math
        y_pred /= K.sum(y_pred, axis=-1, keepdims=True)
        y_pred = K.clip(y_pred, K.epsilon(), 1.0 - K.epsilon())

        # Apply the Weights to the smoothed labels
        weighted_loss = smooth_y_true * K.log(y_pred) * weights
        return -K.sum(weighted_loss, axis=-1)

    return loss

In [ ]:
from sklearn.utils import class_weight
from sklearn.metrics import f1_score
import numpy as np

# flatten the masks so compute_class_weight can see every single pixel
flat_y = Y_train_ready.flatten()
weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(flat_y),
    y=flat_y
)


In [ ]:
print("Class Weights:", weights)

Class Weights: [1.11912802 1.041065   0.75546067 0.97782913 1.32307165 0.95812405]


In [ ]:
# Custom Table Logger
class TableLogger(tf.keras.callbacks.Callback):
    def __init__(self, val_data):
        super().__init__()
        self.X_val, self.y_val_cat = val_data

    def on_train_begin(self, logs=None):
        print(f"\n{'Epoch':<6} | {'Train Loss':<10} | {'Val Loss':<10} | {'Acc':<8} | {'F1 (Macro)':<10}")
        print("-" * 60)

    def on_epoch_end(self, epoch, logs=None):
        val_logits = self.model.predict(self.X_val, verbose=0, batch_size=16)


        val_preds = np.argmax(val_logits, axis=-1).flatten()
        val_true = np.argmax(self.y_val_cat, axis=-1).flatten()

        # Calculate F1
        val_f1 = f1_score(val_true, val_preds, average='macro')

        # Get values from logs
        train_loss = logs.get('loss', 0)
        val_loss = logs.get('val_loss', 0)
        val_acc = logs.get('val_accuracy', 0)

        print(f"{epoch+1:<6} | {train_loss:<10.4f} | {val_loss:<10.4f} | {val_acc:<8.4f} | {val_f1:<10.4f}")


In [ ]:
# Re-build and Compile
model = build_res_unet(input_shape=(256, 256, 7), num_classes=6)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=weighted_categorical_crossentropy(weights, label_smoothing=0.1),
    metrics=['accuracy']
)
# Reduces the learning rate automatically when the model starts "bouncing"
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=3,
    min_lr=1e-7
)

table_logger = TableLogger(val_data=(X_test, Y_test_cat))

# Start training
history = model.fit(
    X_train, Y_train_cat,
    validation_data=(X_test, Y_test_cat),
    epochs=20,
    batch_size=16,
    callbacks=[table_logger, reduce_lr],
    verbose=0
)


Epoch  | Train Loss | Val Loss   | Acc      | F1 (Macro)
------------------------------------------------------------
1      | 2.3721     | 1.7700     | 0.1604   | 0.1167    
2      | 2.0985     | 1.7603     | 0.1791   | 0.1227    
3      | 1.9356     | 1.7660     | 0.2007   | 0.1354    
4      | 1.7666     | 1.7768     | 0.1901   | 0.1369    
5      | 1.6704     | 1.7865     | 0.1834   | 0.1420    
6      | 1.5993     | 1.7888     | 0.1824   | 0.1389    
7      | 1.5837     | 1.7665     | 0.2007   | 0.1551    
8      | 1.5972     | 1.7257     | 0.2538   | 0.1993    
9      | 1.5650     | 1.6631     | 0.3359   | 0.2554    
10     | 1.5550     | 1.5755     | 0.3576   | 0.2912    
11     | 1.5298     | 1.4932     | 0.3973   | 0.3532    
12     | 1.5019     | 1.4240     | 0.4476   | 0.4204    
13     | 1.4758     | 1.3654     | 0.4971   | 0.4774    
14     | 1.4974     | 1.3178     | 0.5342   | 0.5171    
15     | 1.4759     | 1.2809     | 0.5614   | 0.5461    
16     | 1.4506     | 1.25

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, f1_score
import numpy as np

print("Generating predictions...")
train_logits = model.predict(X_train, batch_size=8, verbose=1)
test_logits = model.predict(X_test, batch_size=8, verbose=1)

# Convert probabilities to class indices and FLATTEN to 1D
train_preds = np.argmax(train_logits, axis=-1).flatten()
test_preds = np.argmax(test_logits, axis=-1).flatten()

# Flatten the ground truth labels as well
y_train_flat = Y_train_ready.flatten()
y_test_flat = Y_test_ready.flatten()

# Calculate Overall Metrics
train_acc = accuracy_score(y_train_flat, train_preds)
test_acc = accuracy_score(y_test_flat, test_preds)
test_f1_macro = f1_score(y_test_flat, test_preds, average="macro")

print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy:  {test_acc:.4f}")
print(f"Test Macro F1:  {test_f1_macro:.4f}")

# Detailed Classification Report
merged_names = [
    "Tree cover",
    "Shrubland",
    "Grassland",
    "Cropland/vegetation",
    "Built-up",
    "Permanent water bodies"
]

print("\n--- CLASSIFICATION REPORT ---")
print(classification_report(y_test_flat, test_preds, target_names=merged_names))

Generating predictions...
80/80 ━━━━━━━━━━━━━━━━━━━━ 19s 135ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 6s 421ms/step
Train Accuracy: 0.6014
Test Accuracy:  0.6266
Test Macro F1:  0.6143

--- CLASSIFICATION REPORT ---
                        precision    recall  f1-score   support

            Tree cover       0.68      0.48      0.56    829607
             Shrubland       0.32      0.81      0.46   1016613
             Grassland       0.43      0.21      0.28   1772697
   Cropland/vegetation       0.75      0.70      0.72   1822707
              Built-up       0.86      0.58      0.70   1090768
Permanent water bodies       0.94      0.98      0.96   1659608

              accuracy                           0.63   8192000
             macro avg       0.66      0.63      0.61   8192000
          weighted avg       0.67      0.63      0.62   8192000



In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches
import numpy as np

# Define 6 Merged Names and their corresponding colors
merged_names = [
    "Tree cover",          # Index 0
    "Shrubland",           # Index 1
    "Grassland",           # Index 2
    "Cropland / Bare",     # Index 3
    "Built-up",            # Index 4
    "Water / Wetland"      # Index 5
]

hex_colors = ["#006400", "#ffbb22", "#ffff4c", "#f096ff", "#fa0000", "#0064ff"]
custom_cmap = ListedColormap(hex_colors)

def plot_mask_comparison(X, y_true, model, num_samples=3):
    # Predict on the test samples
    preds_logits = model.predict(X[:num_samples], batch_size=num_samples)
    preds = np.argmax(preds_logits, axis=-1)

    if len(y_true.shape) == 4:
        y_true = np.argmax(y_true, axis=-1)

    fig, axes = plt.subplots(num_samples, 2, figsize=(12, 5 * num_samples))

    if num_samples == 1:
        axes = np.expand_dims(axes, axis=0)

    for i in range(num_samples):
        # Column 1: Actual Mask
        axes[i, 0].imshow(y_true[i], cmap=custom_cmap, vmin=0, vmax=5)
        axes[i, 0].set_title(f"Sample {i+1}: Actual ESA Mask", fontsize=14)
        axes[i, 0].axis('off')

        # Column 2: Model Prediction
        axes[i, 1].imshow(preds[i], cmap=custom_cmap, vmin=0, vmax=5)
        axes[i, 1].set_title(f"Sample {i+1}: ResU-Net Prediction", fontsize=14)
        axes[i, 1].axis('off')

    # Create Legend Patches
    legend_patches = [mpatches.Patch(color=hex_colors[j], label=merged_names[j])
                     for j in range(len(merged_names))]


    fig.legend(handles=legend_patches, loc='center right', borderaxespad=0.1, fontsize=12)
    plt.subplots_adjust(right=0.80, wspace=0.1)

    plt.show()

plot_mask_comparison(X_test, Y_test_cat, model, num_samples=3)